# scripts/ — Test Suite

This notebook:
1. **Discovers** all version sub-folders inside `scripts/` and reads their `metadata.json`.
2. **Reports** metadata details (version, last_modified_at, description, dependencies, etc.).
3. **Validates** the Python script in each version folder using `pytest` (functional tests only — no mocks).

| Cell | Responsibility |
|------|----------------|
| **0** | Environment setup |
| **1** | Version discovery and metadata report |
| **2** | pytest execution |


In [1]:
import subprocess
import sys

_DEPS = ['pytest', 'pytest-timeout']

print('Checking test dependencies...')
for pkg in _DEPS:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    print(f"  {'ok' if r.returncode == 0 else 'FAILED'}: {pkg}")
print()

Checking test dependencies...
  ok: pytest
  ok: pytest-timeout



In [2]:
import json
import pathlib
import datetime

# Resolve the scripts/ root regardless of where the notebook kernel was started.
_NOTEBOOK_DIR = pathlib.Path(globals().get('__vsc_ipynb_file__', __file__ if '__file__' in dir() else '.')).parent
_SCRIPTS_DIR  = _NOTEBOOK_DIR

print(f'scripts/ root : {_SCRIPTS_DIR}')

scripts/ root : c:\Users\evers\Downloads\conversor-lattes-para-curriculum-vitae\scripts


---
## Version discovery and metadata report

In [3]:
def _load_metadata(version_dir: pathlib.Path) -> dict | None:
    """
    Attempt to load metadata.json from a version directory.
    Returns None if the file does not exist or is malformed.
    """
    meta_path = version_dir / 'metadata.json'
    if not meta_path.exists():
        return None
    try:
        with meta_path.open('r', encoding='utf-8') as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        print(f'  Warning: {meta_path} is malformed — {exc}')
        return None


def _discover_versions(scripts_dir: pathlib.Path) -> list[dict]:
    """
    Walk scripts_dir looking for sub-directories named v<N>.
    Returns a list of records sorted by version directory name.
    """
    records = []
    for candidate in sorted(scripts_dir.iterdir()):
        if not candidate.is_dir():
            continue
        # Accept any directory that starts with 'v' followed by a digit.
        if not (candidate.name.startswith('v') and candidate.name[1:].isdigit()):
            continue
        meta = _load_metadata(candidate)
        py_files = sorted(candidate.glob('*.py'))
        records.append({
            'dir'     : candidate,
            'name'    : candidate.name,
            'metadata': meta,
            'py_files': py_files,
        })
    return records


versions = _discover_versions(_SCRIPTS_DIR)
print(f'Versions found: {len(versions)}')
for v in versions:
    has_meta = 'yes' if v['metadata'] else 'no'
    n_py     = len(v['py_files'])
    print(f"  {v['name']}  |  metadata: {has_meta}  |  .py files: {n_py}")

Versions found: 4
  v1  |  metadata: yes  |  .py files: 1
  v2  |  metadata: no  |  .py files: 0
  v3  |  metadata: no  |  .py files: 0
  v4  |  metadata: no  |  .py files: 0


In [4]:
_SEP = '-' * 64


def _format_metadata_report(record: dict) -> str:
    """Render a human-readable summary of a version record."""
    lines = [_SEP, f"Version : {record['name']}"]
    meta  = record['metadata']

    if meta is None:
        lines.append('  No metadata.json found.')
        return '\n'.join(lines)

    fields = [
        ('Script'         , meta.get('script', 'N/A')),
        ('Version'        , meta.get('version', 'N/A')),
        ('Created'        , meta.get('created_at', 'N/A')),
        ('Last modified'  , meta.get('last_modified_at', 'N/A')),
        ('Description'    , meta.get('description', 'N/A')),
        ('Runtime'        , meta.get('runtime', 'N/A')),
        ('Entry point'    , meta.get('entry_point', 'N/A')),
        ('Source notebook', meta.get('source_notebook', 'N/A')),
    ]
    max_key = max(len(k) for k, _ in fields)
    for key, val in fields:
        lines.append(f"  {key:<{max_key}} : {val}")

    # Runtime dependencies
    deps = meta.get('dependencies', {}).get('runtime', [])
    if deps:
        lines.append(f"  {'Runtime deps':<{max_key}} : {', '.join(deps)}")

    # Changelog
    changelog = meta.get('changelog', [])
    if changelog:
        lines.append('  Changelog:')
        for entry in changelog:
            lines.append(f"    [{entry.get('version','?')}] {entry.get('date','?')} — {entry.get('notes','')[:80]}")

    # Known limitations
    limitations = meta.get('known_limitations', [])
    if limitations:
        lines.append('  Known limitations:')
        for lim in limitations:
            lines.append(f"    - {lim[:80]}")

    # Age calculation
    try:
        lm = datetime.date.fromisoformat(meta['last_modified_at'])
        age_days = (datetime.date.today() - lm).days
        lines.append(f"  {'Age (days)':<{max_key}} : {age_days}")
    except (KeyError, ValueError):
        pass

    return '\n'.join(lines)


print('\n=== Metadata Report ===')
for record in versions:
    print(_format_metadata_report(record))
print(_SEP)


=== Metadata Report ===
----------------------------------------------------------------
Version : v1
  Script          : lattes_para_pdf.py
  Version         : 1.0.0
  Created         : 2026-03-05
  Last modified   : 2026-03-05
  Description     : Converts a Lattes CV export (ZIP containing XML) into a professional PDF curriculum vitae. Terminal-based refactor of lattes_para_pdf.ipynb (root). No Jupyter kernel or widget dependencies required.
  Runtime         : python>=3.9
  Entry point     : lattes_para_pdf.py
  Source notebook : ../../lattes_para_pdf.ipynb
  Runtime deps    : reportlab>=4.0.0, tqdm>=4.65.0
  Changelog:
    [1.0.0] 2026-03-05 — Initial release. Full refactor of root-level notebook into a standalone terminal
  Known limitations:
    - Interactive CV editor (Block 5 of the notebook) is not available in terminal mod
    - Times New Roman TTF font registration is Windows-specific. Falls back to built-i
    - Designed for the Lattes XML schema. Other XML schemas are not

---
## Pytest validation

Runs `pytest` against the dedicated test file for each version folder.  
Tests are functional — they exercise real extraction and rendering logic against
the actual test fixture ZIP bundled in the test directory.

> The test file is `scripts/v1/tests/test_lattes_para_pdf.py`.  
> pytest is invoked as a subprocess so its output is captured and displayed here.

In [5]:
def _run_pytest(version_dir: pathlib.Path) -> dict:
    """
    Invoke pytest for a given version directory.
    Looks for a 'tests/' sub-folder; falls back to the directory root.
    Returns a dict with returncode, stdout, stderr.
    """
    tests_subdir = version_dir / 'tests'
    target = str(tests_subdir) if tests_subdir.is_dir() else str(version_dir)

    result = subprocess.run(
        [
            sys.executable, '-m', 'pytest',
            target,
            '-v',
            '--tb=short',
            '--timeout=60',
            '--no-header',
        ],
        capture_output=True,
        text=True,
        cwd=str(version_dir),
    )
    return {
        'returncode': result.returncode,
        'stdout'    : result.stdout,
        'stderr'    : result.stderr,
        'target'    : target,
    }


print('=== pytest Results ===')
all_passed = True

for record in versions:
    ver_name = record['name']
    print(f'\n{_SEP}')
    print(f'Running pytest for {ver_name}...')

    result = _run_pytest(record['dir'])
    status = 'PASSED' if result['returncode'] == 0 else 'FAILED'

    if result['returncode'] != 0:
        all_passed = False

    print(f'  Target : {result["target"]}')
    print(f'  Status : {status} (exit code {result["returncode"]})')

    if result['stdout']:
        print('  Output :')
        for line in result['stdout'].splitlines():
            print(f'    {line}')

    if result['stderr'] and result['returncode'] != 0:
        print('  Stderr :')
        for line in result['stderr'].splitlines():
            print(f'    {line}')

print(f'\n{_SEP}')
print(f'Overall result: {"ALL PASSED" if all_passed else "SOME TESTS FAILED"}')

=== pytest Results ===

----------------------------------------------------------------
Running pytest for v1...
  Target : c:\Users\evers\Downloads\conversor-lattes-para-curriculum-vitae\scripts\v1
  Status : FAILED (exit code 5)
  Output :
    ============================= test session starts =============================
    collecting ... collected 0 items
    
    ============================ no tests ran in 0.01s ============================

----------------------------------------------------------------
Running pytest for v2...
  Target : c:\Users\evers\Downloads\conversor-lattes-para-curriculum-vitae\scripts\v2
  Status : FAILED (exit code 5)
  Output :
    ============================= test session starts =============================
    collecting ... collected 0 items
    
    ============================ no tests ran in 0.01s ============================

----------------------------------------------------------------
Running pytest for v3...
  Target : c:\Users\evers\